# Chapter 4: Understanding Loss Functions and Training Dynamics

**Objective:** To understand the role of loss functions in training a neural network, how to interpret loss values during training, and the process of optimization. This session will also involve analyzing and tweaking the training process to improve model performance.

## 1. What Are Loss Functions?

A loss function measures how well the model’s predictions match the true labels. During training, the optimizer updates the model’s weights to minimize this loss.

**Tasks:**
1. Research the difference between **Cross-Entropy Loss** and **Mean Squared Error (MSE)**. Why is Cross-Entropy Loss more suitable for classification tasks like CIFAR-10?
2. Look at the formula for Cross-Entropy Loss. What does it penalize? How does it handle incorrect predictions with high confidence?

**Questions to Explore:**
* What does a lower loss value mean for the model’s performance?
* How is the loss function connected to the gradients used in backpropagation?

> **Answer:**
> 
> **1. Cross-Entropy vs. MSE:**
> *   **MSE:** Measures squared differences; best for continuous regression tasks.
> *   **Cross-Entropy:** Measures differences between probability distributions; best for classification (like CIFAR-10) as it focuses on the predicted probability of the true class.
> 
> **2. Penalization & Confident Errors:**
> *   It penalizes assigning low probabilities to the correct class. 
> *   For "confidently wrong" predictions (assigning near $0$ probability to the true class), the penalty $-\log(\approx 0)$ approaches infinity.
> 
> **Questions to Explore:**
> *   **Lower Loss Value:** The model predicts the true labels more accurately and with higher certainty.
> *   **Backpropagation:** The loss is the starting point. We calculate its gradients relative to each weight, telling the optimizer how to adjust weights to reduce future errors.

## 2. Training Dynamics

**Tasks:**
1. Look at the training and validation loss/accuracy plots from Chapter 3. Answer the following:
   * Are the training and validation loss decreasing over epochs?
   * Is the gap between training and validation accuracy increasing? What might this indicate?
2. Research the terms **overfitting** and **underfitting**. Based on your plots, is your model showing signs of either? Why?

**Questions to Explore:**
* What might cause the validation accuracy to plateau while training accuracy continues to improve?
* How could you adjust the model or training process to address overfitting or underfitting?

> **Answer:**
> 
> **Tasks:**
> *   1. training loss is continually decreasing (as training accuracy keeps rising), but validation loss likely stopped decreasing after epoch 10 (as validation accuracy plateaus).
> *   2. Yes, a widening gap indicates that the model is overfitting to the training data.
>
> **Questions to Explore:**
> *    1. The model is memorizing training noise and specific details, losing its ability to generalize.
> *    2. Use Early Stopping, Dropout, or data augmentation to mitigate overfitting; increase model complexity to fix underfitting.

## 3. Experimenting with Hyperparameters

Hyperparameters like learning rate, batch size, and number of epochs significantly influence the training process.

**Tasks:**
1. Experiment with different learning rates (e.g., 0.1, 0.01, 0.001). How does a larger or smaller learning rate affect convergence and final accuracy?
2. Vary the batch size (e.g., 32, 64, 128). How does changing the batch size influence training time and performance?
3. Try training the model for fewer or more epochs. At what point does the validation loss start to increase, indicating overfitting?

**Programming Exercise:**
Modify the **model.fit()** call from Chapter 3 to experiment with these hyperparameters. Record your observations for each run.

In [ ]:
# 编程练习：实验不同的超参数

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


# 1. 加载 CIFAR-10 数据集
# 使用在第2章中准备的数据集（包含训练集、验证集和测试集划分）
(train_images_full, train_labels_full), (test_images,
                                         test_labels) = tf.keras.datasets.cifar10.load_data()

# 将像素值归一化到 0 和 1 之间
train_images_full = train_images_full / 255.0
test_images = test_images / 255.0

# 划分训练集和验证集（例如，80% 训练，20% 验证）
train_images, val_images, train_labels, val_labels = train_test_split(
    train_images_full, train_labels_full, test_size=0.2, random_state=42
)


def create_and_train_model(learning_rate, batch_size, epochs):
    print(
        f"\n--- Training with LR: {learning_rate}, Batch Size: {batch_size}, Epochs: {epochs} ---")

    # 1. 重新定义模型 (确保每次实验都从头开始，而不是接着上次的权重训练)
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])

    # 2. 编译模型：这里传入具体的 learning_rate
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # 3. 训练模型：这里传入具体的 batch_size 和 epochs
    history = model.fit(
        train_images,
        train_labels,
        validation_data=(val_images, val_labels),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0  # 设置为0不打印满屏幕的进度条，只看最终结果
    )

    # 打印最终一轮的结果
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    print(
        f"Final Train Accuracy: {final_train_acc:.4f} | Final Val Accuracy: {final_val_acc:.4f}")

    return history


# --- 实验 1: 不同的学习率 (固定 batch_size=64, epochs=15) ---
history_lr_large = create_and_train_model(
    learning_rate=0.1, batch_size=64, epochs=15)
history_lr_good = create_and_train_model(
    learning_rate=0.001, batch_size=64, epochs=15)
history_lr_small = create_and_train_model(
    learning_rate=0.00001, batch_size=64, epochs=15)

# --- 实验 2: 不同的 Batch Size (固定 lr=0.001, epochs=15) ---
history_batch_small = create_and_train_model(
    learning_rate=0.001, batch_size=32, epochs=15)
history_batch_large = create_and_train_model(
    learning_rate=0.001, batch_size=256, epochs=15)


--- Training with LR: 0.1, Batch Size: 64, Epochs: 15 ---


c:\Users\Owner\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Final Train Accuracy: 0.0992 | Final Val Accuracy: 0.1017

--- Training with LR: 0.001, Batch Size: 64, Epochs: 15 ---
Final Train Accuracy: 0.5020 | Final Val Accuracy: 0.4763

--- Training with LR: 1e-05, Batch Size: 64, Epochs: 15 ---
Final Train Accuracy: 0.4126 | Final Val Accuracy: 0.4076

--- Training with LR: 0.001, Batch Size: 32, Epochs: 15 ---
Final Train Accuracy: 0.4903 | Final Val Accuracy: 0.4436

--- Training with LR: 0.001, Batch Size: 256, Epochs: 15 ---
Final Train Accuracy: 0.4853 | Final Val Accuracy: 0.4619


## 4. Regularization Techniques

To improve generalization and prevent overfitting, you can use techniques like dropout and weight regularization.

**Tasks:**
1. Research **dropout layers**. What do they do during training? How can they help prevent overfitting?
2. Research **L2 regularization** (also called weight decay). What effect does it have on the model’s weights?

**Programming Exercise:**
* Add a dropout layer to your model (e.g., Dropout(0.5) after a hidden layer).
* Add L2 regularization to one of the dense layers (e.g., Dense(units, kernel_regularizer=tf.keras.regularizers.l2(0.01))).
* Retrain the model and compare the training and validation accuracy before and after adding these regularization techniques.

> **Answer:**
> 

In [ ]:
# 编程练习：添加正则化技术
# [在此编写代码]

## 5. Reflection on Training Dynamics

After running experiments and analyzing your results, reflect on the following:

**Questions to Reflect On:**
1. How did changing the learning rate affect the model’s ability to converge?
2. What effect did dropout and L2 regularization have on the model’s performance?
3. How do you decide the best combination of hyperparameters for your model?

> **Answer:**
> 

## Programming Assignment:

Write a Python script to:
1. Experiment with at least two different learning rates.
2. Add a dropout layer to your FCNN model.
3. Train the modified model and compare its performance to the original model.

Below is a **skeleton code snippet** for the dropout modification:

In [ ]:
from tensorflow.keras.layers import Dropout, Flatten, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

# 修改模型，添加 Dropout 和 L2 正则化
model = Sequential([
    Flatten(input_shape=(32, 32, 3)),
    # L2 正则化
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),  # Dropout 层
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# 编译并训练模型
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# history = model.fit(
#     train_images,
#     train_labels,
#     validation_data=(val_images, val_labels),
#     epochs=20,
#     batch_size=64
# )